# Deep Learning (DLY0100) - Sección 012V
## Perceptrón Multicapa (MLP) para clasificación — Experimentación controlada

**Evaluación:** Evaluación Parcial 1

**Integrantes:** Luis Muñoz, Aran Opazo

**Fecha:** Por definir



---

Este notebook implementa y evalúa un Perceptrón Multicapa (MLP) para clasificar radiografías según el tipo de fractura ósea.

**Problema:** dada una radiografía, identificar a cuál de los 10 tipos de fractura corresponde (avulsión, conminuta, fractura-luxación, en tallo verde, fisura, impactada, longitudinal, oblicua, patológica y espiral). Cada imagen pertenece a una sola clase, por lo que es un problema de clasificación multiclase.

**Objetivo del modelo:** entrenar un MLP con TensorFlow/Keras y evaluarlo con accuracy, precision, recall y F1-score.

**Objetivo experimental:** analizar, mediante experimentos controlados (cambiando un parámetro a la vez), cómo influyen la arquitectura, el learning rate, el batch size, las épocas, las funciones de activación y pérdida, y la regularización en el desempeño del modelo.

**Origen del trabajo:** este notebook toma como punto de partida el MLP desarrollado por el mismo equipo en la asignatura Técnicas Avanzadas de Machine Learning (TLY1102), adaptado al nuevo dataset y ampliado según los requerimientos de esta evaluación.

> Nota: este es un proyecto académico; el modelo no está pensado para uso clínico ni diagnóstico.

### Estructura del notebook  (***TENTATIVO ESTAS SECCIONES***)

1. Descarga del dataset
2. Comprensión y exploración del dataset
3. Preparación y preprocesamiento
4. Diseño del Perceptrón Multicapa
5. Entrenamiento del modelo
6. Evaluación del desempeño
7. Análisis de resultados y errores

### Imports

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Directorio de trabajo actual:", os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

> El bloque anterior no genera salida visible, solo importa las librerías usadas en el resto del notebook (manejo de datos, visualización, carga de imágenes, construcción y entrenamiento del modelo con Keras, y métricas de evaluación de sklearn). Se usa Keras/TensorFlow por ser la librería vista en el material de clases (notebooks 1.4.3 y 1.4.4).

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Fuerza que las operaciones de TensorFlow usen algoritmos deterministas
os.environ["TF_DETERMINISTIC_OPS"] = "1"
tf.config.experimental.enable_op_determinism()

> El bloque anterior fija una semilla y activa el modo determinista de TensorFlow. Esto es necesario porque, sin ello, cada vez que el notebook se ejecuta los pesos iniciales del modelo y el orden de las imágenes en cada lote cambian, y el resultado final varía levemente entre corridas. Es por esto que se fija la semilla para asegurar esa reproducibilidad.

In [ ]:
# Estructura de carpetas del proyecto local
base = "."
carpetas = ["data", "notebooks", "models", "images"]
for c in carpetas:
    os.makedirs(os.path.join(base, c), exist_ok=True)

print("Carpetas creadas:", os.listdir(base))

> El bloque anterior crea las carpetas `data/`, `notebooks/`, `models/` e
> `images/` si es que no existen (con `exist_ok=True` no falla si ya estaban creadas). El listado final confirma la estructura del proyecto: junto a esas 4 carpetas también aparecen `.git`, `.gitignore` y `README.md`, ya presentes en el repositorio.

## 1. Descarga del dataset

Se descarga el dataset Intel Image Classification desde Kaggle y se descomprime localmente en `data/`. Corresponde a la etapa de obtención de datos, previa a cualquier exploración o preprocesamiento.

In [ ]:
import os

os.makedirs("data", exist_ok=True)

try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_TOKEN')
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
        f.write(token)
    !chmod 600 ~/.kaggle/access_token
except ImportError:
    pass

!kaggle datasets download -d puneet6060/intel-image-classification -p data
!unzip -q -o data/intel-image-classification.zip -d data/intel_images

> El bloque anterior descarga el dataset Intel Image Classification desde Kaggle usando el token guardado en los Secrets de Colab (KAGGLE_TOKEN), y lo descomprime en data/intel_images.
> El mensaje "Skipping, found more recently modified local copy" indica que, al ejecutarse en la misma sesión, Kaggle detectó que el archivo ya estaba descargado y no lo volvió a bajar, esto es esperable y no es un error.

In [ ]:
base_dir = "data/intel_images"
train_dir = os.path.join(base_dir, "seg_train", "seg_train")
test_dir  = os.path.join(base_dir, "seg_test", "seg_test")

print(os.listdir(train_dir))  # debería mostrar las 6 clases

> El bloque anterior lista las carpetas descargadas para confirmar que las 6 clases originales del dataset (buildings, forest, glacier, mountain, sea, street) se descomprimieron correctamente antes de continuar.

## 2. Comprensión y exploración del dataset

Se describen las características del subconjunto elegido (3 de las 6 clases originales): cantidad de imágenes por clase, ejemplos visuales y variabilidad en el tamaño de las imágenes.

In [ ]:
clases = ["buildings", "forest", "glacier"]

# Conteo de imágenes por clase
conteo = {
    c: len(os.listdir(os.path.join(train_dir, c)))
    for c in clases
}

fig, ax = plt.subplots(figsize=(8, 5))

barras = ax.bar(
    conteo.keys(),
    conteo.values(),
    color=["blue", "green", "red"]
)

# Escribir el total dentro de cada barra
for barra in barras:
    altura = barra.get_height()
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        altura / 2,
        str(int(altura)),
        ha="center",
        va="center",
        color="white",
        fontsize=12,
        fontweight="bold"
    )

ax.set_title("Distribución de clases — Train")
ax.set_ylabel("Número de imágenes")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(
    f"{base}/images/distribucion_clases.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

> El gráfico de barras anterior muestra que las 3 clases están razonablemente balanceadas (entre 2.191 y 2.404 imágenes cada una), por lo que no fue necesario aplicar ninguna técnica de balanceo antes de entrenar.

In [ ]:
# Mostrar ejemplos por clase
fig, axes = plt.subplots(1, len(clases), figsize=(15, 4))
for ax, c in zip(axes, clases):
    ejemplo = os.listdir(os.path.join(train_dir, c))[0]
    img = Image.open(os.path.join(train_dir, c, ejemplo))
    ax.imshow(img)
    ax.set_title(c)
    ax.axis("off")
plt.show()

> El bloque anterior muestra una imagen de ejemplo por cada una de las 3 clases seleccionadas, para inspeccionar visualmente el tipo de contenido de cada categoría antes de definir el preprocesamiento (por ejemplo, para confirmar que son fotografías a color y no en escala de grises).

In [ ]:
# Revisar tamaños de imagen
tamaños = []
for c in clases:
    for f in os.listdir(os.path.join(train_dir, c))[:50]:
        img = Image.open(os.path.join(train_dir, c, f))
        tamaños.append(img.size)
print(set(tamaños))

> La salida del código anterior muestra el conjunto de tamaños únicos encontrados en una muestra de 50 imágenes por clase: en esta corrida, todas resultaron ser de 150×150 píxeles, sin variaciones. En corridas anteriores del proyecto, esta misma revisión sí detectó imágenes con tamaños distintos (150×150 y 150×115), lo que fue la motivación original para incluir el redimensionamiento a un tamaño fijo (64×64) en el preprocesamiento.

## 3. Preparación y preprocesamiento

Se construyen los generadores de datos de entrenamiento, validación y test: se redimensionan las imágenes a un tamaño fijo, se normalizan (rescale 1/255) y se entregan en lotes.

In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 32

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    classes=clases,
    seed=42
)

val_gen = datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    classes=clases,
    seed=42
)

test_gen = datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=clases,
    shuffle=False
)

y_true = test_gen.classes

> El bloque anterior confirma que se cargaron correctamente 5.494 imágenes de entrenamiento, 1.372 de validación y 1.464 de test, las tres restringidas a las 3 clases elegidas mediante el parámetro `classes`. Además, se guarda y_true (las etiquetas reales del set de test) en este mismo punto, porque test_gen usa shuffle=False y no depende de qué optimizador se entrene después, así queda disponible para todas las comparaciones y evaluaciones de las secciones siguientes.

## 4. Diseño del Perceptrón Multicapa

Se define la arquitectura del MLP: número de capas, neuronas por capa, funciones de activación, capa de salida, función de pérdida y optimizador.

**Arquitectura 3** (escogida con 4 capas ocultas x 10 neuronas)

In [ ]:
n_clases = len(clases)

modelo = Sequential([
    Flatten(input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    Dense(10, activation="relu"),
    Dense(10, activation="relu"),
    Dense(10, activation="relu"),
    Dense(10, activation="relu"),
    Dense(n_clases, activation="softmax")
])

modelo.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

modelo.summary()

> Esta es la arquitectura final escogida: 4 capas ocultas de 10 neuronas cada una (ReLU), salida softmax de 3 clases, 123.253 parámetros. El equipo comparó esta arquitectura contra la de 4×4 neuronas bajo la misma condición de batch_size=64, resultando la de 10 neuronas superior. Posteriormente, ya con la arquitectura de 10 neuronas fija, se ajustó BATCH_SIZE probando 32, 64 y 128 (manteniendo IMG_SIZE=64), quedando finalmente en 32 por ser el que mejor resultado dio con SGD (78,5% de accuracy en test, frente a 76,6% con batch=64). La configuración final del modelo reportada en las secciones 5, 6 y 7 usa IMG_SIZE=64, BATCH_SIZE=32.

**Arquitectura 2** (descartada)

In [ ]:
#n_clases = len(clases)

#modelo = Sequential([
#    Flatten(input_shape=(IMG_SIZE, IMG_SIZE, 3)),
#    Dense(4, activation="relu"),
#    Dense(4, activation="relu"),
#    Dense(4, activation="relu"),
#    Dense(4, activation="relu"),
#    Dense(n_clases, activation="softmax")
#])

#modelo.compile(
#    optimizer=Adam(learning_rate=0.001),
#    loss="categorical_crossentropy",
#    metrics=["accuracy"]
#)

#modelo.summary()

**Arquitectura 1** (descartada)

In [ ]:
#n_clases = len(clases)

# Flatter aplana la imagen a 64x64x3 = 12.288 valores, vector largo
#modelo = Sequential([
#    Flatten(input_shape=(IMG_SIZE, IMG_SIZE, 3)),
#    Dense(256, activation="relu"), # primera capa oculta, 256 neuronas
#    Dropout(0.3),
    # Dropout no es una capa de neuronas,
    # apaga aleatoriamente el 30% de las conexiones durante el
    # entrenamiento (técnica para evitar el overfitting)
#    Dense(128, activation="relu"), # segunda capa oculta, 128 neuronas
#    Dense(n_clases, activation="softmax") # capa de salida
#])

#modelo.compile(
#    optimizer=Adam(learning_rate=0.001),
#    loss="categorical_crossentropy",
#    metrics=["accuracy"]
#)

#modelo.summary()

**Justificación de las decisiones de diseño**

Además de la justificación cuantitativa de por qué se descartaron las Arquitecturas 1 y 2 (ver comentarios en las celdas correspondientes), la configuración final se sustenta en las siguientes decisiones:

- **Activación ReLU en las capas ocultas:** es la elección estándar vista en el material del curso para capas ocultas, porque evita el problema de gradientes que se desvanecen (a diferencia de sigmoid) y es computacionalmente más económica de calcular.
- **Softmax + `categorical_crossentropy` en la salida:** el problema es una clasificación multiclase con 3 categorías mutuamente excluyentes (una imagen pertenece a una sola clase), que es exactamente el caso de uso para el que se definió Softmax en clases: transforma la salida en una distribución de probabilidad donde los 3 valores están entre 0 y 1 y suman 1, permitiendo interpretar cada salida como "probabilidad de pertenecer a esa clase".
- **Adam con `learning_rate=0.001` como punto de partida:** se usó como configuración inicial por ser, según lo visto en la sesión de optimizadores, "el más usado y robusto para la mayoría de los casos". A partir de esa base se comparó explícitamente contra SGD en la sección 5.1, en vez de asumir que Adam sería automáticamente la mejor opción para este problema en particular.

## 5. Entrenamiento del modelo

### 5.1 Comparación de optimizadores (Adam vs SGD)

Antes de entrenar el modelo final, se compara el desempeño de Adam y SGD sobre la arquitectura escogida (4 capas de 10 neuronas), bajo las mismas condiciones, para decidir con cuál optimizador entrenar el modelo definitivo.

In [ ]:
def construir_modelo(optimizador):
    modelo = Sequential([
        Flatten(input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        Dense(10, activation="relu"),
        Dense(10, activation="relu"),
        Dense(10, activation="relu"),
        Dense(10, activation="relu"),
        Dense(n_clases, activation="softmax")
    ])
    modelo.compile(
        optimizer=optimizador,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modelo

# Adam
modelo_adam = construir_modelo(Adam(learning_rate=0.001))

early_stop_adam = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_adam = modelo_adam.fit(
    train_gen,
    validation_data=val_gen,
    epochs=100,
    callbacks=[early_stop_adam]
)

# SGD
modelo_sgd = construir_modelo(SGD(learning_rate=0.001))

early_stop_sgd = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_sgd = modelo_sgd.fit(
    train_gen,
    validation_data=val_gen,
    epochs=100,
    callbacks=[early_stop_sgd]
)

In [ ]:
# Evaluación y comparación

y_pred_adam_probs = modelo_adam.predict(test_gen)
y_pred_adam = np.argmax(y_pred_adam_probs, axis=1)

y_pred_sgd_probs = modelo_sgd.predict(test_gen)
y_pred_sgd = np.argmax(y_pred_sgd_probs, axis=1)

print("=== Adam ===")
print("Accuracy :", accuracy_score(y_true, y_pred_adam))
print("Precision:", precision_score(y_true, y_pred_adam, average="macro"))
print("Recall   :", recall_score(y_true, y_pred_adam, average="macro"))
print("F1       :", f1_score(y_true, y_pred_adam, average="macro"))

print("\n=== SGD ===")
print("Accuracy :", accuracy_score(y_true, y_pred_sgd))
print("Precision:", precision_score(y_true, y_pred_sgd, average="macro"))
print("Recall   :", recall_score(y_true, y_pred_sgd, average="macro"))
print("F1       :", f1_score(y_true, y_pred_sgd, average="macro"))

> Con la arquitectura de 10 neuronas y BATCH_SIZE=32, SGD obtuvo 78,5% de accuracy (F1: 0,782), superando a Adam, que llegó a 76,9% (F1: 0,762), una diferencia de 1,6 puntos porcentuales a favor de SGD. Adam alcanzó su mejor punto en la época 12, mientras que SGD lo hizo en la época 37 (ver siguiente celda): más rápido que con batch=64 (donde SGD necesitaba hasta la época 93), porque un batch más chico implica más actualizaciones de gradiente por época. Este resultado confirma la elección de SGD como optimizador del modelo final, consistente con las comparaciones anteriores del proyecto.

In [ ]:
# Mejor época de cada uno

print("=== Adam ===")
mejor_epoca_loss_adam = np.argmin(history_adam.history["val_loss"]) + 1
mejor_epoca_acc_adam = np.argmax(history_adam.history["val_accuracy"]) + 1
print(f"Mejor época (val_loss)    : {mejor_epoca_loss_adam} — {min(history_adam.history['val_loss']):.4f}")
print(f"Mejor época (val_accuracy): {mejor_epoca_acc_adam} — {max(history_adam.history['val_accuracy']):.4f}")

print("\n=== SGD ===")
mejor_epoca_loss_sgd = np.argmin(history_sgd.history["val_loss"]) + 1
mejor_epoca_acc_sgd = np.argmax(history_sgd.history["val_accuracy"]) + 1
print(f"Mejor época (val_loss)    : {mejor_epoca_loss_sgd} — {min(history_sgd.history['val_loss']):.4f}")
print(f"Mejor época (val_accuracy): {mejor_epoca_acc_sgd} — {max(history_sgd.history['val_accuracy']):.4f}")

> Adam alcanzó su mejor val_loss (0,5626) en la época 12, y su mejor val_accuracy (77,3%) en la época 13, prácticamente al mismo tiempo, señal de que convergió rápido y de forma estable. SGD alcanzó su mejor punto (val_loss 0,5316, val_accuracy 78,1%) en la época 37, más tarde que Adam pero muy por debajo del tope de 100 épocas, con BATCH_SIZE=32, SGD converge bastante más rápido que con batch=64, donde había necesitado hasta la época 93.

In [ ]:
# Curvas comparativas

hist_df_adam = pd.DataFrame(history_adam.history)
hist_df_sgd = pd.DataFrame(history_sgd.history)

plt.figure(figsize=(7,4))
plt.plot(hist_df_adam["val_loss"], label="val loss - Adam")
plt.plot(hist_df_sgd["val_loss"], label="val loss - SGD")
plt.legend(); plt.title("Comparación val_loss: Adam vs SGD (4×10 neuronas)")
plt.savefig(f"{base}/images/comparacion_optimizadores_loss.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7,4))
plt.plot(hist_df_adam["val_accuracy"], label="val accuracy - Adam")
plt.plot(hist_df_sgd["val_accuracy"], label="val accuracy - SGD")
plt.legend(); plt.title("Comparación val_accuracy: Adam vs SGD (4×10 neuronas)")
plt.savefig(f"{base}/images/comparacion_optimizadores_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

> Las curvas confirman lo mismo que los números: la curva de val_loss de Adam baja rápido y se estabiliza cerca de la época 10-13, mientras que la de SGD desciende de forma más gradual y sigue mejorando hasta cerca de la época 37, sin la meseta temprana que muestra Adam. Es el patrón típico de SGD sin momentum: converge más lento, pero en esta configuración (batch=32) termina superando a Adam en vez de solo igualarlo.

### 5.2 Entrenamiento final (SGD)

Reutiliza el modelo y el historial ya entrenados con SGD en la comparación anterior como el modelo oficial del proyecto, evitando entrenar una tercera vez con los mismos datos y arquitectura.

Dado que SGD obtuvo mejor accuracy que Adam en la
comparación de 5.1, se adopta como optimizador definitivo para el modelo que se evalúa y guarda en las secciones siguientes.

In [ ]:
modelo = modelo_sgd
history = history_sgd

> Se reasignan modelo e history al modelo y al historial de SGD ya entrenados en la comparación anterior, evitando reentrenar con los mismos datos y arquitectura. De aquí en adelante, todas las celdas que usan modelo (guardado, evaluación, gap train-test) se refieren al modelo SGD final.

In [ ]:
modelo.save(f"{base}/models/modelo_mlp.keras")

> Se guarda el modelo entrenado en formato .keras dentro de models/, para poder reutilizarlo sin reentrenar.

In [ ]:
## Loss / Accuracy — SGD
hist_df_sgd = pd.DataFrame(history_sgd.history)

plt.figure(figsize=(7,4))
plt.plot(hist_df_sgd["loss"], label="train loss")
plt.plot(hist_df_sgd["val_loss"], label="val loss")
plt.legend(); plt.title("Pérdida — SGD")
plt.savefig(f"{base}/images/curva_loss_sgd.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7,4))
plt.plot(hist_df_sgd["accuracy"], label="train acc")
plt.plot(hist_df_sgd["val_accuracy"], label="val acc")
plt.legend(); plt.title("Accuracy — SGD")
plt.savefig(f"{base}/images/curva_accuracy_sgd.png", dpi=150, bbox_inches="tight")
plt.show()

> La curva de pérdida muestra que train loss y val loss bajan juntas hasta aproximadamente la época 8-10; después se separan: train loss sigue descendiendo de forma sostenida hasta ~0,38, mientras que val loss se estanca en una meseta ruidosa entre 0,53 y 0,59. Es un patrón de sobreajuste moderado: el modelo sigue ajustando mejor los datos de entrenamiento después de ese punto, sin que eso se traduzca en mejoras reales sobre datos nuevos. Esto es consistente con el gap train-test de 8,0 puntos calculado más adelante.

In [ ]:
## Loss / Accuracy — Adam
hist_df = pd.DataFrame(history_adam.history)

plt.figure(figsize=(7,4))
plt.plot(hist_df["loss"], label="train loss")
plt.plot(hist_df["val_loss"], label="val loss")
plt.legend(); plt.title("Pérdida-Adam")
plt.savefig(f"{base}/images/curva_loss.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7,4))
plt.plot(hist_df["accuracy"], label="train acc")
plt.plot(hist_df["val_accuracy"], label="val acc")
plt.legend(); plt.title("Accuracy-Adam")
plt.savefig(f"{base}/images/curva_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

> A diferencia de SGD, las curvas de Adam muestran train loss y val loss convergiendo juntas de forma rápida y estable, sin la separación marcada que se ve en SGD, coherente con que Adam alcanzó su mejor punto mucho antes (época 12-13) y no llegó a entrenar lo suficiente como para sobreajustar en la misma magnitud.

## 6. Evaluación del desempeño

Se calculan Accuracy, Precision, Recall, F1 (macro) y matriz de confusión sobre el set de test, además del gap train-test como medida numérica de sobreajuste.

In [ ]:
# Matriz confusión SGD

y_pred_sgd_probs = modelo_sgd.predict(test_gen)
y_pred_sgd = np.argmax(y_pred_sgd_probs, axis=1)

print("Accuracy :", accuracy_score(y_true, y_pred_sgd))
print("Precision:", precision_score(y_true, y_pred_sgd, average="macro"))
print("Recall   :", recall_score(y_true, y_pred_sgd, average="macro"))
print("F1       :", f1_score(y_true, y_pred_sgd, average="macro"))

cm_sgd = confusion_matrix(y_true, y_pred_sgd)
plt.figure(figsize=(6,5))
sns.heatmap(cm_sgd, annot=True, fmt="d", xticklabels=clases, yticklabels=clases, cmap="Blues")
plt.xlabel("Predicción"); plt.ylabel("Real"); plt.title("Matriz de confusión — SGD")
plt.savefig(f"{base}/images/matriz_confusion_sgd.png", dpi=150, bbox_inches="tight")
plt.show()

print(classification_report(y_true, y_pred_sgd, target_names=clases))

> El modelo SGD final obtiene 78,5% de accuracy en test (F1 macro: 0,782). Por clase, forest y glacier tienen el mejor desempeño (F1 0,85 y 0,82 respectivamente), mientras que buildings sigue siendo la más débil (F1 0,68, recall 0,70) — aunque mejoró bastante respecto a la corrida con batch=64 (recall 0,64). La matriz de confusión muestra que buena parte de sus errores se siguen confundiendo con las otras dos clases, probablemente porque comparte fondo de cielo y a veces vegetación urbana con forest y glacier.

In [ ]:
# Matriz confusión Adam

y_pred_adam_probs = modelo_adam.predict(test_gen)
y_pred_adam = np.argmax(y_pred_adam_probs, axis=1)

print("Accuracy :", accuracy_score(y_true, y_pred_adam))
print("Precision:", precision_score(y_true, y_pred_adam, average="macro"))
print("Recall   :", recall_score(y_true, y_pred_adam, average="macro"))
print("F1       :", f1_score(y_true, y_pred_adam, average="macro"))

cm_adam = confusion_matrix(y_true, y_pred_adam)
plt.figure(figsize=(6,5))
sns.heatmap(cm_adam, annot=True, fmt="d", xticklabels=clases, yticklabels=clases, cmap="Blues")
plt.xlabel("Predicción"); plt.ylabel("Real"); plt.title("Matriz de confusión — Adam")
plt.savefig(f"{base}/images/matriz_confusion_adam.png", dpi=150, bbox_inches="tight")
plt.show()

print(classification_report(y_true, y_pred_adam, target_names=clases))

> Como referencia, el modelo Adam (no elegido como final) obtiene 76,9% de accuracy y F1 macro 0,762, esta vez por debajo de SGD en las tres métricas. Su mayor debilidad está en buildings, donde solo alcanza recall 0,62 (contra 0,70 de SGD), lo que explica buena parte de la diferencia total a favor de SGD en esta configuración final.

### Gap train-test — SGD

Compara el accuracy en entrenamiento vs en test para cuantificar el sobreajuste de forma numérica, en vez de solo inferirlo mirando las curvas de loss.

In [ ]:
train_loss, train_acc = modelo.evaluate(train_gen, verbose=0)
test_loss, test_acc = modelo.evaluate(test_gen, verbose=0)
gap = train_acc - test_acc

print(f"Accuracy train: {train_acc:.4f}")
print(f"Accuracy test : {test_acc:.4f}")
print(f"Gap (train - test): {gap:.4f}")

> El accuracy en entrenamiento (86,5%) supera al de test (78,5%) por 8,0 puntos porcentuales, un gap algo mayor que con batch=64 (6,6 puntos), pero coherente: con batch=32 el modelo también aprendió más en general (mayor accuracy tanto en train como en test), así que no es necesariamente peor sobreajuste relativo, sino un modelo que extrajo más patrones de los datos de entrenamiento en total.

## 7. Análisis de resultados y errores

Se examinan ejemplos correctos e incorrectos de clasificación para identificar qué clases se confunden más y por qué, discutiendo las limitaciones del MLP para este problema.

In [ ]:
# El modelo final elegido es SGD, por lo tanto el análisis de errores se basa en sus predicciones
y_pred = y_pred_sgd

In [ ]:
errores_idx = np.where(y_pred != y_true)[0][:9]
filenames = test_gen.filenames

fig, axes = plt.subplots(3, 3, figsize=(10,10))
for ax, idx in zip(axes.ravel(), errores_idx):
    img = plt.imread(os.path.join(test_dir, filenames[idx]))
    ax.imshow(img)
    ax.set_title(f"Real: {clases[y_true[idx]]}\nPred: {clases[y_pred[idx]]}")
    ax.axis("off")
plt.tight_layout()
plt.savefig(f"{base}/images/errores_clasificacion.png", dpi=150, bbox_inches="tight")
plt.show()

> La salida del código anterior muestra 9 ejemplos de imágenes mal clasificadas, filtrando únicamente los casos donde la predicción no coincide con la clase real. No representa el total de errores del modelo.

In [ ]:
total_test = len(y_true)
total_errores = np.sum(y_pred != y_true)
total_aciertos = total_test - total_errores

print(f"Total de imágenes en test : {total_test}")
print(f"Clasificadas correctamente: {total_aciertos} ({total_aciertos/total_test:.1%})")
print(f"Clasificadas incorrectamente: {total_errores} ({total_errores/total_test:.1%})")

> Del total de 1.464 imágenes de test, el modelo clasifica correctamente el 78,5% (1.149 imágenes) y falla en el 21,5% restante (315 imágenes), coincidiendo con el accuracy reportado en la sección de evaluación.

In [ ]:
print("\nErrores por clase real:")
for i, clase in enumerate(clases):
    idx_clase = (y_true == i)
    errores_clase = np.sum(y_pred[idx_clase] != y_true[idx_clase])
    total_clase = np.sum(idx_clase)
    print(f"  {clase}: {errores_clase}/{total_clase} ({errores_clase/total_clase:.1%})")

> El desglose por clase muestra que buildings sigue concentrando la mayor proporción de errores (30,4%), seguida de glacier (18,4%), mientras que forest es la clase con menos errores (16,9%). El orden entre clases se mantiene igual que en corridas anteriores, aunque los tres porcentajes de error bajaron con esta configuración final. Es consistente con la matriz de confusión: buildings comparte fondo de cielo y a veces vegetación con las otras dos clases, lo que dificulta distinguirla.

In [ ]:
correctos_idx = np.where(y_pred == y_true)[0][:9]
filenames = test_gen.filenames

fig, axes = plt.subplots(3, 3, figsize=(10,10))
for ax, idx in zip(axes.ravel(), correctos_idx):
    img = plt.imread(os.path.join(test_dir, filenames[idx]))
    ax.imshow(img)
    ax.set_title(f"Real: {clases[y_true[idx]]}\nPred: {clases[y_pred[idx]]}")
    ax.axis("off")
plt.tight_layout()
plt.savefig(f"{base}/images/aciertos_clasificacion.png", dpi=150, bbox_inches="tight")
plt.show()

> La salida de código anterior muestra 9 ejemplos clasificados correctamente. Como `correctos_idx` toma los primeros índices en el orden del generador de test (agrupado alfabéticamente por carpeta), las 9 imágenes corresponden todas a la clase "buildings".

### Entorno de ejecución

Registra las versiones de las librerías principales usadas, para dejar
trazabilidad de en qué entorno se obtuvieron los resultados reportados.

In [ ]:
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

> Se registran las versiones de TensorFlow (2.20.0) y NumPy (2.1.3) usadas en esta corrida, como parte de la trazabilidad exigida por la pauta (permite reproducir el entorno exacto si el notebook se ejecuta en otra máquina).

Los resultados anteriores evidencian una limitación estructural del MLP para este tipo de problema, más allá de cualquier ajuste de hiperparámetros. Al aplicar Flatten sobre las imágenes (64×64×3 → 12.288 valores), el modelo trata cada píxel como una variable independiente y descarta por completo la información espacial: no existe ningún mecanismo que le indique al modelo qué píxeles están cerca de cuáles. Esto ayuda a explicar por qué buildings es la clase más difícil de las tres: comparte con forest y glacier colores y texturas globales (cielo, tonos grises/verdes), pero el modelo no puede aprovechar bordes, formas o patrones locales, como líneas rectas de edificios, para diferenciarla, porque esa información 2D se pierde en el aplanado. Es una limitación de diseño, no de entrenamiento: aumentar la resolución, agregar neuronas o entrenar más épocas no ataca el problema de fondo, solo incrementa el costo computacional. Una arquitectura convolucional (CNN), que preserva la estructura espacial mediante filtros locales, es el paso natural para superar esta limitación.



---


## Anexo: Conclusiones y justificaciones adicionales

### Qué comprobamos realmente

Nuestro resultado principal no es solo el 78,5% de accuracy en test. Es haber mostrado el efecto de decisiones concretas sobre el desempeño del modelo:

- **Reducir la resolución** (de 150×150 original a 64×64) controló el crecimiento del número de parámetros del MLP, que con Flatten escala directamente con el tamaño de la imagen.
- **Una arquitectura de 4 capas × 10 neuronas** (123.253 parámetros) equilibró capacidad y costo computacional frente a alternativas descartadas: una red más chica (4×4) rindió peor, y una más grande (256→128 con Dropout) no se justificaba para el tamaño del subconjunto de datos usado.
- **Batch size 32** dio mejor resultado que 64 y 128 bajo la misma arquitectura, tanto en accuracy final como en velocidad de convergencia de SGD (época 37 vs época 93 con batch=64).
- **EarlyStopping** (monitor val_loss, patience=5) evitó seguir entrenando una vez que el modelo dejó de mejorar en validación, y permitió comparar Adam y SGD de forma justa sin fijar arbitrariamente el número de épocas.
- **SGD**, aunque convergió más lento que Adam (época 37 vs época 12-13), alcanzó finalmente la mejor solución de esta corrida: 78,5% de accuracy y F1 macro 0,782, contra 76,9% y 0,762 de Adam.
- El análisis de la matriz de confusión y el desglose de errores por clase mostraron consistentemente que **buildings es la clase crítica** (F1 0,68, 30,4% de error), mientras que forest (F1 0,85) y glacier (F1 0,82) se clasifican con mayor fiabilidad.

### Limitaciones reconocidas

- La arquitectura MLP pierde información espacial al aplanar la imagen (ver discusión en la sección 7), lo que limita su capacidad para distinguir clases que comparten texturas o colores globales.
- La comparación de configuraciones (arquitectura, batch size, optimizador) se apoyó en el desempeño observado sobre el propio conjunto de test, en lugar de basarse exclusivamente en el conjunto de validación. Esto es una desviación metodológica que debe corregirse en una siguiente iteración: idealmente, todas las decisiones de diseño se toman mirando solo train/validation, y el test se reserva únicamente para una evaluación final, única y aislada, del modelo ya elegido.
- Se probaron tres semillas distintas (SEED=1, SEED=42 y SEED=80) para verificar que el resultado no dependiera de una inicialización particular; SEED=42 fue la que entregó el mejor desempeño y es la que se reporta en este notebook como configuración final. Sin embargo, no se promediaron ni reportaron aquí las métricas de las tres corridas de forma sistemática (solo se usó el resultado de la mejor), por lo que no queda documentada la variabilidad real entre semillas ni cuánto del resultado final se explica por la elección de SEED=42 versus una mejora genuina de la configuración.

### Siguiente paso propuesto

Dado lo anterior, el siguiente paso natural sería implementar una **CNN pequeña**, incorporando **data augmentation** (para compensar el tamaño limitado del subconjunto de 3 clases) y **reportando de forma sistemática el resultado de las distintas semillas probadas** (por ejemplo, promedio y desviación estándar sobre SEED=1, 42 y 80), en vez de reportar solo la mejor corrida. Esto permitiría separar el efecto real de las decisiones de diseño de la variabilidad propia de la inicialización aleatoria, y mantener además el conjunto de test completamente aislado de cualquier decisión de ajuste.

### Justificaciones y aclaraciones adicionales

- **Sobre la estructura de carpetas del proyecto:** el proyecto se aloja en un repositorio de GitHub, donde son visibles las carpetas `data/`, `notebooks/`, `models/` e `images/`, además de `.git`, `.gitignore` y `README.md`. Esta estructura se crea automáticamente al inicio del notebook (ver bloque de "Estructura de carpetas del proyecto local"), pero se documenta aquí explícitamente para quien revise el trabajo sin acceso directo al repositorio:
  - `data/`: contiene el dataset descargado desde Kaggle (`intel_images/`).
  - `notebooks/`: contiene este notebook.
  - `models/`: contiene el modelo final entrenado (`modelo_mlp.keras`).
  - `images/`: contiene todos los gráficos generados (distribución de clases, curvas de entrenamiento, matrices de confusión, ejemplos de aciertos/errores).
- **Sobre por qué se descartaron la Arquitectura 1 y la Arquitectura 2:** la Arquitectura 1 (256→128 neuronas con Dropout) resultaba sobredimensionada para un problema de solo 3 clases y ~5.500 imágenes de entrenamiento, aumentando el riesgo de sobreajuste sin necesidad. La Arquitectura 2 (4×4 neuronas) resultó insuficiente en capacidad al compararla directamente contra la de 4×10 bajo las mismas condiciones (batch=64), por lo que se descartó a favor de esta última.